In [ ]:
# ============================================================
# NOTEBOOK 07D — FINAL EXPERIMENTAL ALIGNMENT
# TalTech semantic EDT / representation-model comparison
#
# PURPOSE
# 1) Freeze ONE common set of (UUID, timestamp, target) samples
#    shared by Tabular, Sequential and Graph pipelines.
# 2) Build the PRIMARY tabular representation from EXACTLY the
#    same 5x3 sensor history used by Sequential/Graph:
#       [t-60,t-45,t-30,t-15,t] x [T,RH,CO2] = 15 values.
#    Calendar features are NOT used in the primary comparison.
# 3) Keep the 07B sequential tensors intact; only whole sequences
#    not belonging to the common sample set are removed.
# 4) Filter graph snapshots to the same common node-level keys.
# 5) Replace the shuffled graph control by a degree-preserving
#    rewiring of G_operational.
#
# NO MODEL IS TRAINED HERE.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().resolve()
O = ROOT / "outputs"

COMMON_FILE = O / "06_common_experiment" / "01_common_supervised_dataset.csv"
TABULAR_07A = O / "07A_tabular" / "01_tabular_all_unscaled.csv"
SEQ_DIR = O / "07B_sequential_aligned"
GRAPH_DIR = O / "07C_graph"
OUT = O / "07D_final_alignment"
OUT.mkdir(parents=True, exist_ok=True)

FEATURES = ["temperature", "relative_humidity", "co2"]
TARGET = "target_temperature_t_plus_15"
OFFSETS = [-60, -45, -30, -15, 0]
SPLITS = ["train", "validation", "test"]
SEED = 42

for p in [COMMON_FILE, TABULAR_07A, SEQ_DIR, GRAPH_DIR]:
    assert p.exists(), f"Missing required input: {p}"

print("=" * 72)
print("NOTEBOOK 07D — FINAL EXPERIMENTAL ALIGNMENT")
print("=" * 72)

# ------------------------------------------------------------
# 1. COMMON SOURCE + CURRENT 07A KEYS
# ------------------------------------------------------------
common = pd.read_csv(COMMON_FILE)
common["timestamp"] = pd.to_datetime(common["timestamp"], utc=True)
common["target_timestamp"] = pd.to_datetime(common["target_timestamp"], utc=True)
for c in FEATURES + [TARGET]:
    common[c] = pd.to_numeric(common[c], errors="coerce")

tab_old = pd.read_csv(TABULAR_07A)
tab_old["timestamp"] = pd.to_datetime(tab_old["timestamp"], utc=True)
if "target_timestamp" in tab_old:
    tab_old["target_timestamp"] = pd.to_datetime(tab_old["target_timestamp"], utc=True)

# Current 07A/07B already share these node-level keys.
ab_keys = tab_old[["uuid", "timestamp", "split"]].drop_duplicates()

# ------------------------------------------------------------
# 2. GRAPH-ELIGIBLE NODE-LEVEL KEYS
# ------------------------------------------------------------
graph_meta_parts = []
for s in SPLITS:
    p = GRAPH_DIR / f"metadata_graph_{s}.csv"
    assert p.exists(), f"Missing {p}"
    gm = pd.read_csv(p)
    gm["sequence_end"] = pd.to_datetime(gm["sequence_end"], utc=True)
    gm["target_timestamp"] = pd.to_datetime(gm["target_timestamp"], utc=True)
    gm["split"] = s
    graph_meta_parts.append(gm)

graph_meta = pd.concat(graph_meta_parts, ignore_index=True)

node_index = pd.read_csv(GRAPH_DIR / "01_graph_node_index.csv")
devices = node_index.sort_values("node_index")["uuid"].tolist()
assert len(devices) == 7

graph_keys = (
    graph_meta[["split", "sequence_end"]]
    .rename(columns={"sequence_end": "timestamp"})
    .assign(_k=1)
    .merge(pd.DataFrame({"uuid": devices, "_k": 1}), on="_k")
    .drop(columns="_k")
)

# Intersection = definitive experimental sample universe.
final_keys = ab_keys.merge(
    graph_keys,
    on=["uuid", "timestamp", "split"],
    how="inner",
    validate="one_to_one"
).sort_values(["split", "timestamp", "uuid"]).reset_index(drop=True)

counts = final_keys["split"].value_counts().reindex(SPLITS).fillna(0).astype(int)

print("\nFINAL COMMON NODE-LEVEL COUNTS")
print(counts.to_string())

# Expected from current pipeline
expected = {"train": 64701, "validation": 13825, "test": 13790}
actual = counts.to_dict()
assert actual == expected, f"Unexpected final counts: {actual}; expected {expected}"

final_keys.to_csv(OUT / "01_final_common_sample_keys.csv", index=False)

# ------------------------------------------------------------
# 3. BUILD PRIMARY TABULAR CORE = SAME 5x3 INFORMATION
# ------------------------------------------------------------
# We build directly from common source, with strict same split + segment.
# Feature ordering is time-major, exactly mirroring a 5x3 sequence:
# t-60:[T,RH,CO2], t-45:[T,RH,CO2], ..., t:[T,RH,CO2].

src = common.set_index(["uuid", "timestamp"]).sort_index()

rows = []
feature_names = []
for off in OFFSETS:
    suffix = "t" if off == 0 else f"t_minus_{abs(off)}"
    for v in FEATURES:
        feature_names.append(f"{v}_{suffix}")

for r in final_keys.itertuples(index=False):
    u, end_t, split = r.uuid, r.timestamp, r.split
    end_row = src.loc[(u, end_t)]
    if isinstance(end_row, pd.DataFrame):
        raise AssertionError(f"Duplicate source row: {u}, {end_t}")

    seg = end_row["segment_id"]
    values = []
    for off in OFFSETS:
        t = end_t + pd.Timedelta(minutes=off)
        rr = src.loc[(u, t)]
        if isinstance(rr, pd.DataFrame):
            raise AssertionError(f"Duplicate history row: {u}, {t}")
        assert rr["split"] == split, "Tabular history crosses split."
        assert rr["segment_id"] == seg, "Tabular history crosses segment."
        values.extend([float(rr[v]) for v in FEATURES])

    out = {
        "uuid": u,
        "timestamp": end_t,
        "target_timestamp": end_row["target_timestamp"],
        "split": split,
        "segment_id": seg,
        TARGET: float(end_row[TARGET]),
    }
    out.update(dict(zip(feature_names, values)))
    rows.append(out)

tab_core = pd.DataFrame(rows)
assert len(tab_core) == len(final_keys)
assert np.isfinite(tab_core[feature_names].to_numpy()).all()

tab_core.to_csv(OUT / "02_tabular_core_15_unscaled.csv", index=False)

# Train-only feature-wise scaler for Ridge.
scaler_tab = StandardScaler()
train_mask = tab_core["split"].eq("train")
scaler_tab.fit(tab_core.loc[train_mask, feature_names])

tab_scaled = tab_core.copy()
tab_scaled[feature_names] = scaler_tab.transform(tab_core[feature_names])
tab_scaled.to_csv(OUT / "03_tabular_core_15_scaled.csv", index=False)

pd.DataFrame({
    "feature": feature_names,
    "train_mean": scaler_tab.mean_,
    "train_scale": scaler_tab.scale_
}).to_csv(OUT / "04_tabular_core_scaler.csv", index=False)

# Preserve old 19-feature representation ONLY as secondary sensitivity data.
old19 = tab_old.merge(final_keys, on=["uuid", "timestamp", "split"], how="inner")
old19.to_csv(OUT / "05_tabular_calendar_augmented_secondary.csv", index=False)

# ------------------------------------------------------------
# 4. FILTER 07B SEQUENCES — NEVER ALTER SEQUENCE CONTENT
# ------------------------------------------------------------
seq_meta_all = pd.read_csv(SEQ_DIR / "01_all_sequence_metadata.csv")
seq_meta_all["timestamp"] = pd.to_datetime(seq_meta_all["timestamp"], utc=True)
seq_meta_all["sequence_start"] = pd.to_datetime(seq_meta_all["sequence_start"], utc=True)
seq_meta_all["target_timestamp"] = pd.to_datetime(seq_meta_all["target_timestamp"], utc=True)

seq_outputs = []
seq_audit = []

for s in SPLITS:
    meta_path = SEQ_DIR / f"metadata_{s}.csv"
    Xp = SEQ_DIR / f"X_{s}_unscaled.npy"
    yp = SEQ_DIR / f"y_{s}.npy"
    assert meta_path.exists() and Xp.exists() and yp.exists()

    m = pd.read_csv(meta_path)
    for c in ["timestamp", "sequence_start", "target_timestamp"]:
        m[c] = pd.to_datetime(m[c], utc=True)

    X = np.load(Xp)
    y = np.load(yp)
    assert len(m) == len(X) == len(y)

    fk = final_keys[final_keys["split"] == s][["uuid", "timestamp"]].copy()
    fk["_keep"] = True

    mm = m.reset_index(names="_old_index").merge(
        fk, on=["uuid", "timestamp"], how="left", validate="one_to_one"
    )
    keep = mm["_keep"].eq(True).to_numpy(dtype=bool)
    idx = mm.loc[keep, "_old_index"].to_numpy(dtype=int)

    Xf, yf = X[idx], y[idx]
    mf = m.iloc[idx].reset_index(drop=True)

    # Integrity unchanged
    assert Xf.shape[1:] == (5, 3)
    dur = (mf["timestamp"] - mf["sequence_start"]).dt.total_seconds() / 60
    dtar = (mf["target_timestamp"] - mf["timestamp"]).dt.total_seconds() / 60
    assert (dur == 60).all() and (dtar == 15).all()

    np.save(OUT / f"X_sequence_{s}_unscaled.npy", Xf)
    np.save(OUT / f"y_sequence_{s}.npy", yf)
    mf.to_csv(OUT / f"metadata_sequence_{s}.csv", index=False)

    seq_outputs.append((s, Xf, yf, mf))
    seq_audit.append({"split": s, "sequences": len(mf), "shape": str(Xf.shape)})

# Refit sequential 3-channel scaler on FINAL train only.
Xtrain = next(x[1] for x in seq_outputs if x[0] == "train")
sc_seq = StandardScaler()
sc_seq.fit(Xtrain.reshape(-1, 3))

for s, Xf, yf, mf in seq_outputs:
    Xs = sc_seq.transform(Xf.reshape(-1, 3)).reshape(Xf.shape).astype(np.float32)
    np.save(OUT / f"X_sequence_{s}_scaled.npy", Xs)

pd.DataFrame({
    "feature": FEATURES,
    "train_mean": sc_seq.mean_,
    "train_scale": sc_seq.scale_
}).to_csv(OUT / "06_sequence_scaler.csv", index=False)

# ------------------------------------------------------------
# 5. FILTER GRAPH SNAPSHOTS
# ------------------------------------------------------------
graph_audit = []
for s in SPLITS:
    gm = pd.read_csv(GRAPH_DIR / f"metadata_graph_{s}.csv")
    gm["sequence_end"] = pd.to_datetime(gm["sequence_end"], utc=True)

    Xg = np.load(GRAPH_DIR / f"X_graph_{s}_unscaled.npy")
    yg = np.load(GRAPH_DIR / f"y_graph_{s}.npy")
    assert len(gm) == len(Xg) == len(yg)

    # Since final keys are defined from graph eligibility, all current
    # graph snapshots should remain. Still verify explicitly.
    wanted_times = set(
        final_keys[final_keys["split"] == s]["timestamp"].unique()
    )
    keep = gm["sequence_end"].isin(wanted_times).to_numpy()
    Xgf, ygf = Xg[keep], yg[keep]
    gmf = gm.loc[keep].reset_index(drop=True)

    assert Xgf.shape[1:] == (5, 7, 3)

    np.save(OUT / f"X_graph_{s}_unscaled.npy", Xgf)
    np.save(OUT / f"y_graph_{s}.npy", ygf)
    gmf.to_csv(OUT / f"metadata_graph_{s}.csv", index=False)

    graph_audit.append({
        "split": s,
        "snapshots": len(gmf),
        "node_predictions": len(gmf) * 7,
        "shape": str(Xgf.shape)
    })

# Graph train-only 3-channel scaler on FINAL graph train.
Xg_train = np.load(OUT / "X_graph_train_unscaled.npy")
gmean = Xg_train.reshape(-1, 3).mean(axis=0)
gstd = Xg_train.reshape(-1, 3).std(axis=0)
gstd[gstd == 0] = 1

for s in SPLITS:
    Xgf = np.load(OUT / f"X_graph_{s}_unscaled.npy")
    Xgs = ((Xgf - gmean) / gstd).astype(np.float32)
    np.save(OUT / f"X_graph_{s}_scaled.npy", Xgs)

pd.DataFrame({
    "feature": FEATURES,
    "train_mean": gmean,
    "train_scale": gstd
}).to_csv(OUT / "07_graph_scaler.csv", index=False)

# ------------------------------------------------------------
# 6. DEGREE-PRESERVING SHUFFLED CONTROL
# ------------------------------------------------------------
edges = pd.read_csv(GRAPH_DIR / "05_operational_graph_edges.csv")
op = {tuple(sorted((int(r.source_index), int(r.target_index))))
      for r in edges.itertuples(index=False)}

def degree_dict(es, n=7):
    d = {i: 0 for i in range(n)}
    for a, b in es:
        d[a] += 1; d[b] += 1
    return d

orig_degree = degree_dict(op)
rng = np.random.default_rng(SEED)

# Double-edge swaps preserve every node's degree.
def degree_preserving_rewire(edge_set, swaps=200, max_attempts=20000):
    E = set(edge_set)
    successful = 0
    attempts = 0
    while successful < swaps and attempts < max_attempts:
        attempts += 1
        e = list(E)
        i, j = rng.choice(len(e), size=2, replace=False)
        a, b = e[i]
        c, d = e[j]
        if len({a,b,c,d}) < 4:
            continue
        # randomly choose one of two swap patterns
        if rng.random() < 0.5:
            n1, n2 = tuple(sorted((a,d))), tuple(sorted((c,b)))
        else:
            n1, n2 = tuple(sorted((a,c))), tuple(sorted((b,d)))
        if n1[0] == n1[1] or n2[0] == n2[1]:
            continue
        if n1 in E or n2 in E or n1 == n2:
            continue
        E.remove((a,b)); E.remove((c,d))
        E.add(n1); E.add(n2)
        successful += 1
    return E, successful

# For a very small/dense 7-node graph, a standard random double-edge
# swap can legitimately fail to find a distinct realization even after
# many attempts. We therefore enumerate all simple graphs with the same
# number of edges and retain those with EXACTLY the same labeled degree
# sequence. This is exhaustive here because C(21,10)=352716 only.

import itertools

all_possible_edges = [
    (i, j) for i in range(7) for j in range(i + 1, 7)
]

candidates = []
for comb in itertools.combinations(all_possible_edges, len(op)):
    cand = set(comb)
    if cand == op:
        continue
    if degree_dict(cand) == orig_degree:
        candidates.append(cand)

if candidates:
    # Deterministic choice under the declared seed.
    rng2 = np.random.default_rng(SEED)
    rewired = candidates[int(rng2.integers(len(candidates)))]
    nswaps = None
    shuffled_method = "exhaustive_same_labeled_degree_sequence"
else:
    # Important scientific fallback: do NOT invent a degree-preserving
    # control if the observed labeled degree sequence has a unique
    # realization. Keep the original random same-edge-count control and
    # report explicitly that degree preservation is impossible here.
    old_shuf = pd.read_csv(GRAPH_DIR / "07_shuffled_graph_edges.csv")
    rewired = {
        tuple(sorted((int(r.source_index), int(r.target_index))))
        for r in old_shuf.itertuples(index=False)
    }
    assert len(rewired) == len(op)
    nswaps = None
    shuffled_method = "same_node_and_edge_count_only_degree_preservation_not_available"

assert rewired != op
assert len(rewired) == len(op)

degree_preserved = (degree_dict(rewired) == orig_degree)

rewired_df = pd.DataFrame([
    {
        "source_index": a, "target_index": b,
        "source_uuid": devices[a], "target_uuid": devices[b],
        "relation": "degree_preserving_shuffled_control"
    }
    for a,b in sorted(rewired)
])
rewired_df.to_csv(OUT / "08_shuffled_degree_preserving_edges.csv", index=False)

def edge_index(es):
    directed = []
    for a,b in sorted(es):
        directed.extend([(a,b),(b,a)])
    return np.asarray(directed, dtype=np.int64).T

np.save(OUT / "edge_index_operational.npy", edge_index(op))
np.save(OUT / "edge_index_shuffled_degree_preserving.npy", edge_index(rewired))
np.save(OUT / "edge_index_native.npy", np.empty((2,0), dtype=np.int64))

degree_audit = pd.DataFrame({
    "node_index": range(7),
    "uuid": devices,
    "degree_operational": [orig_degree[i] for i in range(7)],
    "degree_shuffled": [degree_dict(rewired)[i] for i in range(7)]
})
degree_audit.to_csv(OUT / "09_degree_preservation_audit.csv", index=False)

# ------------------------------------------------------------
# 7. CROSS-REPRESENTATION TARGET/KEY VALIDATION
# ------------------------------------------------------------
# Tabular vs sequential exact key equality.
tab_keys_final = set(zip(tab_core.uuid, tab_core.timestamp, tab_core.split))
seq_keys_final = set()
for s, Xf, yf, mf in seq_outputs:
    seq_keys_final |= set(zip(mf.uuid, mf.timestamp, mf.split))
assert tab_keys_final == seq_keys_final

# Every graph snapshot corresponds to all 7 final node-level keys.
graph_expanded = set()
for s in SPLITS:
    gm = pd.read_csv(OUT / f"metadata_graph_{s}.csv")
    gm["sequence_end"] = pd.to_datetime(gm["sequence_end"], utc=True)
    for t in gm["sequence_end"]:
        for u in devices:
            graph_expanded.add((u, t, s))
assert graph_expanded == tab_keys_final

# ------------------------------------------------------------
# 8. AUDIT + FROZEN PROTOCOL
# ------------------------------------------------------------
audit = pd.DataFrame({
    "split": SPLITS,
    "tabular_samples": [int(counts[s]) for s in SPLITS],
    "sequential_samples": [next(len(x[3]) for x in seq_outputs if x[0]==s) for s in SPLITS],
    "graph_snapshots": [next(x["snapshots"] for x in graph_audit if x["split"]==s) for s in SPLITS],
    "graph_node_predictions": [next(x["node_predictions"] for x in graph_audit if x["split"]==s) for s in SPLITS],
})
audit.to_csv(OUT / "10_final_alignment_audit.csv", index=False)

protocol = {
    "status": "FROZEN_BEFORE_MODEL_TRAINING",
    "common_unit_for_metrics": "node-level prediction: (UUID, timestamp, target_temperature_t_plus_15)",
    "final_counts": {s: int(counts[s]) for s in SPLITS},
    "history": "60 min; offsets [-60,-45,-30,-15,0]",
    "forecast_horizon": "15 min",
    "primary_information_content": "T/RH/CO2 at five time steps = 15 sensor values",
    "tabular_primary": "flattened 5x3 history; 15 features; no calendar features",
    "tabular_calendar_augmented": "secondary sensitivity analysis only",
    "sequential": "same 5x3 history, unchanged sequence content",
    "graph": "same 5x3 history per each of 7 UUID nodes",
    "graph_topology": "operational kNN k=2 inferred from training only",
    "spatial_claim": False,
    "uuid_to_ifc_space_mapping": False,
    "graph_controls": [
        "native/no inferred inter-device edges",
        "operational/train-only kNN",
        "degree-preserving shuffled control"
    ],
    "shuffled_control_method": shuffled_method,
    "degree_sequence_preserved": bool(degree_preserved),
    "degree_preserving_alternative_graphs_found": int(len(candidates)),
    "scaling": "fit on final training partition only; targets unscaled",
    "validation_test_used_for_topology": False
}
with (OUT / "11_FROZEN_EXPERIMENTAL_PROTOCOL.json").open("w", encoding="utf-8") as f:
    json.dump(protocol, f, indent=2)

print("\n" + "="*72)
print("FINAL ALIGNMENT AUDIT")
print("="*72)
print(audit.to_string(index=False))

print("\nDEGREE-PRESERVING CONTROL")
print(degree_audit.to_string(index=False))
print("Shuffled control method:", shuffled_method)
print("Degree sequence preserved:", degree_preserved)
print("Distinct degree-preserving alternatives found:", len(candidates))

print("\nPRIMARY TABULAR FEATURES:", len(feature_names))
print(feature_names)

print("\nVALIDATION")
print("✓ Exact same node-level samples for Tabular / Sequential / Graph")
print("✓ Train=64701, Validation=13825, Test=13790")
print("✓ Primary Tabular uses exactly the same 5x3 sensor information")
print("✓ Sequential sequences remain intact")
print("✓ Graph history remains 5x7x3")
print("✓ Operational topology remains train-only")
if degree_preserved:
    print("✓ Shuffled graph preserves every node degree")
else:
    print("! Exact degree-preserving alternative is unavailable for this graph")
    print("✓ Shuffled control preserves node count and edge count; limitation recorded")
print("✓ No spatial UUID->IfcSpace claim")
print("✓ Protocol frozen before model training")
